# CIFAR-10 CNN 분류기

In [31]:
import torch

In [32]:
from typing import Tuple

from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## 1. 데이터셋

In [33]:
# CIFAR-10: 32x32 컬러(3채널), 10개 클래스
# 학습셋에만 증강을 주고, 정규화는 양쪽 동일하게
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train = datasets.CIFAR10(root="./cifar_data", train=True, download=True, transform=train_transform)
cifar_test = datasets.CIFAR10(root="./cifar_data", train=False, download=True, transform=test_transform)

# 모델 생성 셀에서 len(label_names) 로 쓰이는 값
label_names = cifar_train.classes
print(label_names)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [34]:
print(len(cifar_train), len(cifar_test))
print(cifar_train[0][0].shape, cifar_train[0][1])  # torch.Size([3, 32, 32]) 6

50000 10000
torch.Size([3, 32, 32]) 6


## 2. DataLoader

In [35]:
batch_size = 128
train_loader = DataLoader(cifar_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(cifar_test, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)  # torch.Size([128, 3, 32, 32]) torch.Size([128])

torch.Size([128, 3, 32, 32]) torch.Size([128])


## 3. 모델

conv/pool 를 거치며 feature map 이 이렇게 줄어든다:

`(3,32,32)` -> conv_1 -> pool_1 `(32,16,16)` -> conv_2 -> pool_2 `(64,8,8)`
-> conv_3 (padding 없음) `(128,6,6)` -> pool_3 `(128,3,3)` -> conv_4 (padding 없음) `(128,1,1)`
-> flatten `(128,)` -> fc_1 -> fc_2 `(10,)`

In [36]:
class MLPClassifier(nn.Module):
    def __init__(self, output_dim: int):
        super(MLPClassifier, self).__init__()

        # 3*3*32
        self.input_dimension = 3 * 32 * 32

        self.fc_1 = nn.Linear(self.input_dimension, 128)
        self.fc_2 = nn.Linear(128, 128)
        self.fc_3 = nn.Linear(128, output_dim, bias=True)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #-1에는 batch_size가 들어가고, 3*32*32=3072가 두 번째 차원에 들어간다.
        x = x.view(-1, self.input_dimension)
    
        x = self.fc_1(x)
        x = self.dropout(F.relu(x))

        x = self.fc_2(x)
        x = self.dropout(F.relu(x))
        
        x = self.fc_3(x)
        return x

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epoch = 20
learning_rate = 1e-3
lr_decay = 0.9

model = MLPClassifier(len(label_names)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, lr_decay)
criterion = nn.CrossEntropyLoss()

In [38]:
def evaluate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, float]:
    avg_loss = torch.tensor(0.0, device=device)
    avg_accuracy = torch.tensor(0.0, device=device)

    # eval(): dropout을 끈다 / no_grad(): 평가에는 역전파가 필요 없으니 그래프를 만들지 않는다
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            input_images, labels = batch
            input_images, labels = input_images.to(device), labels.to(device)

            logits = model(input_images)
            predicted = torch.argmax(logits, dim=1)
            loss = criterion(logits, labels)

            avg_accuracy += (predicted == labels).sum()
            # .item() 은 배치마다 GPU->CPU 동기화가 걸리므로 텐서로 누적한다
            avg_loss += loss * input_images.size(0)

    avg_accuracy = (avg_accuracy / len(dataloader.dataset)).item()
    avg_loss = (avg_loss / len(dataloader.dataset)).item()

    return avg_loss, avg_accuracy

In [39]:
def train_one_epoch(model: nn.Module, dataloader: DataLoader, criterion: nn.Module,
                    optimizer: torch.optim.Optimizer, device: torch.device) -> Tuple[float, float]:
    avg_loss = torch.tensor(0.0, device=device)
    avg_accuracy = torch.tensor(0.0, device=device)

    model.train()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        # detach 안 하면 epoch 내내 그래프가 안 풀려서 메모리가 쌓인다
        avg_loss += loss.detach() * images.size(0)
        avg_accuracy += (logits.detach().argmax(dim=1) == labels).sum()

    num_samples = len(dataloader.dataset)
    return (avg_loss / num_samples).item(), (avg_accuracy / num_samples).item()

In [40]:
for epoch in range(1, num_epoch + 1):
    current_lr = lr_scheduler.get_last_lr()[0]
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    lr_scheduler.step()

    print(f"[Epoch {epoch}/{num_epoch}] lr={current_lr:.4e} | "
          f"train loss {train_loss:.4f}, acc {train_acc*100:.2f}% | "
          f"test loss {test_loss:.4f}, acc {test_acc*100:.2f}%")

[Epoch 1/20] lr=1.0000e-03 | train loss 1.9583, acc 28.67% | test loss 1.7164, acc 38.19%
[Epoch 2/20] lr=9.0000e-04 | train loss 1.8274, acc 34.07% | test loss 1.6690, acc 40.57%
[Epoch 3/20] lr=8.1000e-04 | train loss 1.7753, acc 36.11% | test loss 1.6494, acc 42.16%
[Epoch 4/20] lr=7.2900e-04 | train loss 1.7412, acc 37.20% | test loss 1.6414, acc 42.12%
[Epoch 5/20] lr=6.5610e-04 | train loss 1.7087, acc 38.29% | test loss 1.6275, acc 43.07%
[Epoch 6/20] lr=5.9049e-04 | train loss 1.6830, acc 39.40% | test loss 1.6031, acc 43.94%
[Epoch 7/20] lr=5.3144e-04 | train loss 1.6657, acc 40.02% | test loss 1.5848, acc 44.58%
[Epoch 8/20] lr=4.7830e-04 | train loss 1.6454, acc 40.72% | test loss 1.5745, acc 44.76%
[Epoch 9/20] lr=4.3047e-04 | train loss 1.6346, acc 41.41% | test loss 1.5526, acc 45.42%
[Epoch 10/20] lr=3.8742e-04 | train loss 1.6233, acc 42.00% | test loss 1.5549, acc 45.26%
[Epoch 11/20] lr=3.4868e-04 | train loss 1.6122, acc 42.20% | test loss 1.5505, acc 45.27%
[Epoch 1

In [41]:
sum(p.numel() for p in model.parameters())

411146

In [42]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu()

print("정답:", [label_names[i] for i in labels[:10].tolist()])
print("예측:", [label_names[i] for i in preds[:10].tolist()])

정답: ['cat', 'ship', 'ship', 'airplane', 'frog', 'frog', 'automobile', 'frog', 'cat', 'automobile']
예측: ['cat', 'truck', 'ship', 'ship', 'deer', 'cat', 'cat', 'frog', 'dog', 'automobile']
